---
title: "Lab: Real-Time Streaming Pipeline with Kafka & Delta Lake"
format: html
---


# Day 2 — Real-Time Data Pipelines

This single notebook covers everything in today's lab:

1. **Mock Kafka streaming pipeline** — producers, a stateful window processor
   (tumbling/sliding/watermark), and a Delta Lake ACID sink.
2. **Real Kafka round trip** — one real `kafka-python` producer/consumer exchange with a
   Pydantic schema gate, run against a real local broker.


In [ ]:
!pip install pyspark==3.5.0 delta-spark==3.2.0


## Part 1 — Mock Kafka Streaming Pipeline (windowing + Delta sink)

In [ ]:
import asyncio
import os
import random
import shutil
from datetime import datetime, timedelta, timezone
from collections import defaultdict
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)
from delta import configure_spark_with_delta_pip

In [ ]:
# =====================================================================
# 1. BACKBONE — Mock Kafka Broker (partitions + offsets)

In [ ]:
# =====================================================================

class MockKafkaBroker:
    """
    Simulates Apache Kafka's core model.

    In real Kafka:
      - A topic is split into N partitions (ordered, immutable logs).
      - Each message gets an offset — its address within the partition.
      - A consumer group shares partitions; each partition goes to
        exactly one member, so adding consumers beyond the partition
        count yields no extra throughput.
      - Committed offsets let consumers resume after a crash.
    """

    def __init__(self):
        self.topics  = defaultdict(asyncio.Queue)
        self.offsets = defaultdict(int)

    async def publish(self, topic: str, message: dict):
        offset = self.offsets[topic]
        self.offsets[topic] += 1
        await self.topics[topic].put({
            "offset":     offset,
            "event_time": datetime.now(timezone.utc).isoformat(),
            "topic":      topic,
            "payload":    message,
        })

    async def consume(self, topic: str) -> dict:
        return await self.topics[topic].get()


def init_delta_spark() -> SparkSession:
    builder = (
        SparkSession.builder
        .appName("Day2_RealTimeStreamingPipeline")
        .master("local[*]")
        .config("spark.sql.extensions",
                "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.warehouse.dir", "./data/warehouse")
    )
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    return spark

In [ ]:
# =====================================================================
# 2. INGESTION LAYER — Three concurrent IoT producers

In [ ]:
# =====================================================================

async def machine_producer(broker: MockKafkaBroker, machine_id: str):
    """
    Simulates a factory sensor streaming temperature readings every 300ms.
    12% chance of a spike anomaly to trigger downstream alerts.
    """
    print(f"  🏭 [PRODUCER] {machine_id} stream online.")
    while True:
        is_spike = random.random() < 0.12
        temp = random.uniform(92.0, 118.0) if is_spike else random.uniform(58.0, 78.0)
        await broker.publish("factory_sensors_raw", {
            "machine_id": machine_id,
            "temperature": round(temp, 2),
            "event_ts":   datetime.now(timezone.utc).isoformat(),
        })
        await asyncio.sleep(0.3)

In [ ]:
# =====================================================================
# 3. COMPUTE LAYER — Stream processor with window types + watermark

In [ ]:
# =====================================================================

class StreamProcessor:
    """
    Stateful stream processing engine — mirrors Flink / Spark Structured Streaming.

    Three window types demonstrated:
      TUMBLING   — fixed, non-overlapping 1-second buckets.
                   Every event belongs to exactly one bucket.
                   Use case: per-second event count, hourly totals.

      SLIDING    — overlapping rolling window over the last 3 seconds.
                   The same event can appear in multiple windows.
                   Use case: rolling average temperature or CPU.

      WATERMARK  — events older than WATERMARK_S seconds are dropped.
                   This bounds state memory and lets the engine close
                   windows cleanly. Larger watermark = more completeness
                   but higher latency.
    """

    WATERMARK_S      = 2
    SLIDING_WINDOW_S = 3

    def __init__(self, broker: MockKafkaBroker, threshold: float):
        self.broker    = broker
        self.threshold = threshold
        self.sliding   = defaultdict(list)
        self.tumbling  = defaultdict(list)
        self.watermark = datetime.now(timezone.utc)

    async def run(self):
        print(f"  🧠 [PROCESSOR] Window engine live. Threshold: {self.threshold}°C")
        while True:
            event   = await self.broker.consume("factory_sensors_raw")
            payload = event["payload"]
            m_id    = payload["machine_id"]
            val     = payload["temperature"]
            now     = datetime.now(timezone.utc)

            # Watermark — drop late events
            if now < self.watermark - timedelta(seconds=self.WATERMARK_S):
                print(f"  ⚠️  [WATERMARK] Late event from {m_id} dropped.")
                continue
            self.watermark = max(self.watermark, now)

            # Sliding window — rolling average
            self.sliding[m_id].append((now, val))
            cutoff = now - timedelta(seconds=self.SLIDING_WINDOW_S)
            self.sliding[m_id] = [
                (t, v) for t, v in self.sliding[m_id] if t > cutoff
            ]
            rolling_avg = (
                sum(v for _, v in self.sliding[m_id]) / len(self.sliding[m_id])
            )

            # Tumbling window — count events in this 1-second bucket
            bucket = now.strftime("%H:%M:%S")
            self.tumbling[bucket].append(val)
            bucket_count = len(self.tumbling[bucket])

            # Threshold check
            if val > self.threshold:
                await self.broker.publish("anomalies_sink", {
                    "machine_id":   m_id,
                    "value":        val,
                    "rolling_avg":  round(rolling_avg, 2),
                    "bucket":       bucket,
                    "bucket_count": bucket_count,
                    "status":       "CRITICAL",
                })

In [ ]:
# =====================================================================
# 4. STORAGE SINK — ACID append to Delta Lake

In [ ]:
# =====================================================================

async def delta_sink(
    broker: MockKafkaBroker,
    spark: SparkSession,
    delta_path: str,
):
    """
    Reads anomaly alerts and commits them to Delta Lake.
    Delta's transaction log guarantees ACID — no partial writes,
    no corruption even if two writers race on the same files.
    """
    schema = StructType([
        StructField("event_time",   StringType(),  True),
        StructField("machine_id",   StringType(),  True),
        StructField("value",        DoubleType(),  True),
        StructField("rolling_avg",  DoubleType(),  True),
        StructField("bucket",       StringType(),  True),
        StructField("bucket_count", IntegerType(), True),
        StructField("status",       StringType(),  True),
    ])
    print("  💾 [SINK] Delta Lake writer active.")
    while True:
        alert = await broker.consume("anomalies_sink")
        p = alert["payload"]
        row = [(
            alert["event_time"],
            p["machine_id"],
            p["value"],
            p["rolling_avg"],
            p["bucket"],
            p["bucket_count"],
            p["status"],
        )]
        spark.createDataFrame(row, schema).write \
             .format("delta").mode("append").save(delta_path)
        print(f"  🚨 [DELTA WRITE] {p['machine_id']} — "
              f"{p['value']}°C | rolling avg {p['rolling_avg']}°C | "
              f"bucket {p['bucket']} ({p['bucket_count']} events)")

In [ ]:
# =====================================================================
# 5. PIPELINE ORCHESTRATION

In [ ]:
# =====================================================================

async def main():
    delta_path = "./data/delta/anomalies"

    if os.path.exists("./data"):
        shutil.rmtree("./data")

    broker = MockKafkaBroker()
    spark  = init_delta_spark()

    print("\n🚀 Real-Time Streaming Pipeline booting...\n")

    processor = asyncio.create_task(
        StreamProcessor(broker, threshold=85.0).run()
    )
    sink = asyncio.create_task(
        delta_sink(broker, spark, delta_path)
    )
    producers = [
        asyncio.create_task(machine_producer(broker, "CNC_MILL_01")),
        asyncio.create_task(machine_producer(broker, "ROBOTIC_ARM_02")),
        asyncio.create_task(machine_producer(broker, "ASSEMBLY_LINE_03")),
    ]

    print("  Streaming for 8 seconds...\n")
    await asyncio.sleep(8.0)

    for task in [processor, sink, *producers]:
        task.cancel()

    print("\n📊 Final Delta Lake state (all committed anomalies):")
    if os.path.exists(delta_path):
        spark.read.format("delta").load(delta_path) \
             .orderBy("event_time").show(truncate=False)
    else:
        print("  No anomalies recorded in this run — lower the threshold to see more.")

    spark.stop()
    print("\n🏁 Pipeline shut down cleanly.")

## ▶ Run

In [ ]:
await main()


🚀 Real-Time Streaming Pipeline booting...

  Streaming for 8 seconds...

  🧠 [PROCESSOR] Window engine live. Threshold: 85.0°C
  💾 [SINK] Delta Lake writer active.
  🏭 [PRODUCER] CNC_MILL_01 stream online.
  🏭 [PRODUCER] ROBOTIC_ARM_02 stream online.
  🏭 [PRODUCER] ASSEMBLY_LINE_03 stream online.
  🚨 [DELTA WRITE] CNC_MILL_01 — 113.9°C | rolling avg 113.9°C | bucket 21:32:12 (1 events)

📊 Final Delta Lake state (all committed anomalies):
+--------------------------------+-----------+-----+-----------+--------+------------+--------+
|event_time                      |machine_id |value|rolling_avg|bucket  |bucket_count|status  |
+--------------------------------+-----------+-----+-----------+--------+------------+--------+
|2026-06-27T21:32:12.761585+00:00|CNC_MILL_01|113.9|113.9      |21:32:12|1           |CRITICAL|
+--------------------------------+-----------+-----+-----------+--------+------------+--------+


🏁 Pipeline shut down cleanly.


## Part 2 — Real Kafka Round Trip

Part 1's `MockKafkaBroker` is kept because it makes windowing logic easy to read and step
through, but it never touches the actual Kafka client library. This part does one real
`KafkaProducer` -> topic -> `KafkaConsumer` round trip with a Pydantic schema gate that
rejects a deliberately malformed message instead of crashing.

Needs a running Kafka broker. In Colab, run this in a cell first:
```
!pip install kafka-python pydantic
# Run a local single-broker Kafka in the background (Colab has Java preinstalled):
!curl -sSOL https://downloads.apache.org/kafka/3.7.0/kafka_2.13-3.7.0.tgz && tar -xzf kafka_2.13-3.7.0.tgz
!cd kafka_2.13-3.7.0 && bin/kafka-storage.sh format -t $(bin/kafka-storage.sh random-uuid) -c config/kraft/server.properties
!cd kafka_2.13-3.7.0 && nohup bin/kafka-server-start.sh config/kraft/server.properties > /tmp/kafka.log 2>&1 &
```
Give it ~15-20 seconds to finish starting before running the cells below.


In [1]:
!pip install kafka-python pydantic



[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import time

from pydantic import BaseModel, ValidationError, field_validator

TOPIC = "factory_sensors_raw"
BOOTSTRAP_SERVERS = "localhost:9092"


class SensorReading(BaseModel):
    """Same shape the mock pipeline's producers emit — reused here as the schema gate."""

    machine_id: str
    temperature: float
    event_ts: str

    @field_validator("temperature")
    @classmethod
    def plausible_range(cls, v: float) -> float:
        if not (-40.0 <= v <= 200.0):
            raise ValueError(f"temperature {v} outside plausible sensor range")
        return v


def produce_sample_messages():
    from kafka import KafkaProducer

    producer = KafkaProducer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    )

    good = [
        {"machine_id": "CNC_MILL_01", "temperature": 71.4, "event_ts": "2026-06-29T10:00:00Z"},
        {"machine_id": "ROBOTIC_ARM_02", "temperature": 68.9, "event_ts": "2026-06-29T10:00:01Z"},
    ]
    # A malformed message on purpose, to prove the consumer's schema gate catches it
    # instead of crashing the pipeline.
    bad = {"machine_id": "ASSEMBLY_LINE_03", "temperature": "not_a_number", "event_ts": "2026-06-29T10:00:02Z"}

    for msg in good + [bad]:
        producer.send(TOPIC, msg)
        print(f"  [PRODUCER] sent -> {msg}")
    producer.flush()
    producer.close()


def consume_and_validate(max_messages: int = 3, timeout_s: int = 10):
    from kafka import KafkaConsumer

    consumer = KafkaConsumer(
        TOPIC,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        auto_offset_reset="earliest",
        enable_auto_commit=True,
        group_id="day2-schema-check",
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=timeout_s * 1000,
    )

    accepted, rejected = 0, 0
    for record in consumer:
        try:
            reading = SensorReading.model_validate(record.value)
            accepted += 1
            print(f"  [CONSUMER] accepted @offset {record.offset}: {reading}")
        except ValidationError as e:
            rejected += 1
            print(f"  [CONSUMER] REJECTED @offset {record.offset}: {e.errors()[0]['msg']}")
        if accepted + rejected >= max_messages:
            break
    consumer.close()
    print(f"\n  Summary: {accepted} accepted, {rejected} rejected by the schema gate.")


def main():
    print("Real Kafka round trip (topic: {})".format(TOPIC))
    try:
        produce_sample_messages()
        time.sleep(1)
        consume_and_validate()
    except ImportError as e:
        print(f"\nMissing dependency: {e}")
        print("Install it first: pip install -r requirements.txt")
    except Exception as e:
        print(f"\nCould not reach Kafka at {BOOTSTRAP_SERVERS}: {e}")
        print("Start a local broker first (see the docker run command in this file's docstring),")
        print("then re-run this cell. Part 1 above (the mock-broker pipeline)")
        print("still works without Kafka running.")


In [3]:
main()


Real Kafka round trip (topic: factory_sensors_raw)

Could not reach Kafka at localhost:9092: KafkaTimeoutError: Unable to bootstrap from localhost:9092
Start a local broker first (see the docker run command in this file's docstring),
then re-run this cell. Part 1 above (the mock-broker pipeline)
still works without Kafka running.
